<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **5. Wi-Fi Contention Protocol**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 5.1 Automatic Repeat Request (ARQ)

Wireless packets can be lost through interference, collisions, or weak signals. Automatic Repeat Request (ARQ) uses acknowledgments (ACKs) and retransmissions to recover missing data. This notebook first compares retransmission strategies, then examines how devices share a radio channel.

Stop-and-Wait ARQ sends one packet and waits for its ACK. If the ACK does not arrive before a timeout, the sender retransmits the packet. The small amount of state makes this approach simple, but waiting for each ACK can leave the channel unused.

Go-Back-N ARQ allows several packets to be outstanding within a sending window. In the basic scheme, the receiver accepts packets in order. When a packet is missing, the sender retransmits it and the later outstanding packets, including any that already reached the receiver out of order.

Selective Repeat ARQ retransmits only packets that remain unacknowledged. The receiver can buffer packets that arrive out of order. This can avoid unnecessary retransmissions, at the cost of more buffering and acknowledgment bookkeeping.

An absent ACK does not identify the cause of failure: the data packet or the ACK itself may have been lost.


#### 5.1.1 Illustrative ARQ comparison

The code varies packet-loss probability and compares three simplified retransmission routines. Its throughput score includes manually chosen penalties for retransmissions and waiting. It does not simulate packet timing, ACK traffic, or a complete protocol state machine, so the plotted percentages are illustrative scores rather than measured channel utilization. The Go-Back-N routine is also an approximation of window recovery.


In [ ]:
import matplotlib.pyplot as plt
import random

import numpy as np
import math
from scipy.stats import binom

In [ ]:
NUM_PACKETS = 30000  # Number of packets to send
WINDOW_SIZE = 10    # Window size for Go-Back-N and Selective Repeat
LOSS_PROBS = [i / 100 for i in range(0, 51, 5)]  # Packet loss probabilities from 0 to 50%

def simulate_stop_and_wait(num_packets, loss_prob):
    transmitted = 0
    retransmissions = 0
    successful = 0

    for i in range(num_packets):
        transmitted += 1
        while random.random() < loss_prob:
            retransmissions += 1
            transmitted += 1
        successful += 1

    throughput = successful /  (transmitted+retransmissions*1.2) / 1.4
    return throughput

def simulate_go_back_n(num_packets, window_size, loss_prob):
    transmitted = 0
    retransmissions = 0
    successful = 0
    base = 0
    next_seq = 0

    while successful < num_packets:
        while next_seq < base + window_size and next_seq < num_packets:
            transmitted += 1
            if random.random() < loss_prob:
                retransmissions += next_seq - base
                next_seq = base
            else:
                next_seq += 1
                successful += 1
        base = next_seq

    throughput = successful / (transmitted+retransmissions*1.2)
    return throughput

def simulate_selective_repeat(num_packets, window_size, loss_prob):
    transmitted = 0
    retransmissions = 0
    successful = 0
    base = 0
    acked = [False] * num_packets

    while successful < num_packets:
        for i in range(base, min(base + window_size, num_packets)):
            if not acked[i]:
                transmitted += 1
                if random.random() < loss_prob:
                    retransmissions += 1
                else:
                    acked[i] = True
                    successful += 1
        while base < num_packets and acked[base]:
            base += 1

    throughput = successful / (transmitted+retransmissions*1.2)
    return throughput

# Collect data for plotting
sw_throughputs = []
gbn_throughputs = []
sr_throughputs = []

for prob in LOSS_PROBS:
    sw_throughputs.append(100*simulate_stop_and_wait(NUM_PACKETS, prob))
    gbn_throughputs.append(100*simulate_go_back_n(NUM_PACKETS, WINDOW_SIZE, prob))
    sr_throughputs.append(100*simulate_selective_repeat(NUM_PACKETS, WINDOW_SIZE, prob))

#### 5.1.2 Plot the illustrative throughput scores


In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
theLineWidth = 2

ax.cla()
ax.axis([None, None, None, None])

plt.plot(LOSS_PROBS, sw_throughputs, label='Stop-and-Wait', linewidth=theLineWidth)
plt.plot(LOSS_PROBS, gbn_throughputs, label='Go-Back-N (Window Size=' + str(WINDOW_SIZE) + ')', linewidth=theLineWidth)    
plt.plot(LOSS_PROBS, sr_throughputs, label='Selective Repeat (Window Size=' + str(WINDOW_SIZE) + ')', linewidth=theLineWidth)
plt.xlabel('Packet Loss Probability')
plt.ylabel('Throughput [%]')
plt.grid(True)
plt.xlim(0, 0.5)
plt.ylim(0, 100)
ax.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper right", facecolor=(.6, 0.61, 0.8))
plt.tight_layout()



### 5.2 Access protocols

An access protocol decides when a device may transmit on a shared channel. ALOHA and carrier-sense protocols illustrate different ways of handling competing transmissions.

#### 5.2.1 Pure ALOHA

ALOHA originated in the University of Hawaii's early packet radio network. In pure ALOHA, a device transmits whenever a packet is ready. Overlapping transmissions may collide. If no ACK arrives, the device waits a random time before trying again.

For equal-duration packets and Poisson-distributed transmission attempts, the idealized throughput is $S=G e^{-2G}$. Here $G$ is the mean number of attempts per packet duration and $S$ is the mean number of successful packets per packet duration. A packet is vulnerable to another transmission starting during a two-packet-duration interval. The maximum is $1/(2e)\approx18.4\%$ of one successful packet per packet duration.

#### 5.2.2 Slotted ALOHA

Slotted ALOHA restricts transmissions to the start of synchronized slots, each one packet long. Two or more transmissions in the same slot collide in the idealized model. Random waiting before retrying helps prevent repeated collisions.

Slot alignment reduces the vulnerable interval to one packet duration. Under the same traffic assumptions, $S=G e^{-G}$, with a maximum of $1/e\approx36.8\%$. These percentages describe the idealized models, with ACK overhead and other protocol details omitted.

#### 5.2.3 Carrier Sense Multiple Access (CSMA)

Carrier Sense Multiple Access (CSMA) checks whether the channel is busy before transmitting. Its variants differ in how a device waits and retries. A 1-persistent device transmits as soon as it senses an idle channel; a non-persistent device waits a random time before checking again after finding it busy. A p-persistent scheme uses a probability $p$ to decide whether to transmit at an eligible idle slot.

Carrier sensing cannot prevent every collision. Signals take time to propagate, and a device may not hear another transmitter that can still interfere at the receiver. The latter situation is the hidden-terminal problem.

CSMA with Collision Detection (CSMA/CD) was used on shared, half-duplex Ethernet links. A transmitter detects a collision, stops, and retries after a random delay. Modern switched, full-duplex Ethernet links do not use this collision mechanism.

Wi-Fi uses CSMA with Collision Avoidance (CSMA/CA), including random backoff and acknowledgments for ordinary unicast traffic. A backoff counter counts down during eligible idle slots and freezes while the medium is busy. A missing ACK can trigger a retry, but does not prove that a collision occurred.

#### 5.2.4 Comparison

| Protocol | Access rule | Main limitation |
|---|---|---|
| Pure ALOHA | Transmit when a packet is ready | Overlapping packets collide; idealized peak utilization is 18.4% |
| Slotted ALOHA | Transmit at a slot boundary | Requires synchronization; idealized peak utilization is 36.8% |
| CSMA | Sense the channel before attempting transmission | Performance depends on propagation delay, backoff, traffic, and hidden terminals |

The ALOHA limits are useful reference points. CSMA has no single corresponding efficiency percentage that applies to every network.


#### 5.2.5 Simulate access protocols

The code generates random transmission attempts and counts successful packets. Packet duration is normalized to one time unit. The CSMA routines use a sensing delay of 0.1 packet durations and simplified handling of pending traffic. They illustrate differences between access rules; they do not implement the full Ethernet or Wi-Fi protocols.


In [ ]:
def slotted_aloha(arrival_times: np.ndarray):
    count_arrivals = np.zeros(math.ceil(arrival_times[-1]) + 1)
    for val in arrival_times:
        count_arrivals[math.ceil(val)] += 1
    return np.sum(1 * (count_arrivals == 1))


def pure_aloha(inter_arrival_time: np.ndarray):
    success = 0
    for j in range(len(inter_arrival_time)):
        if j - 1 >= 0 and inter_arrival_time[j - 1] >= 1 and j + 1 < N and inter_arrival_time[j + 1] >= 1:
            success += 1
    return success


def csma_generalized(arrival_times: np.ndarray, p, cd: bool = False):
    beta = .1
    success = 0
    transmission_ends = 0
    collision_detect_possible = 0
    successfully_sending = False
    pending_packets = 0
    for j, time in enumerate(arrival_times):
        if time < collision_detect_possible:
            # channel is busy, but detected as free due to beta delay in detection of ongoing transmission
            if cd:
                transmission_ends = min(transmission_ends, time + beta)
            else:
                transmission_ends = time + 1
            if successfully_sending:
                success -= 1
                successfully_sending = False
        elif time < transmission_ends:
            # channel is busy, do nothing
            pending_packets += 1
            if (j + 1 < len(arrival_times) and arrival_times[j + 1] >= transmission_ends) \
                    or j + 1 == len(arrival_times):
                if p == 1:
                    x = pending_packets
                elif p == 0:
                    x = 0
                else:
                    x = binom(pending_packets, p).rvs()
                if x == 1:
                    success += 1
                    successfully_sending = True
                    collision_detect_possible = transmission_ends + beta
                    transmission_ends = transmission_ends + 1
                elif x > 1:
                    collision_detect_possible = transmission_ends + beta
                    transmission_ends = transmission_ends + 1
                pending_packets = 0
        else:
            # channel is free
            successfully_sending = True
            success += 1
            collision_detect_possible = time + beta
            transmission_ends = time + 1
    return success


N = int(10000)
numG = 30
rng = np.random.default_rng()
throughput_pure = np.zeros(numG)
throughput_slotted = np.zeros(numG)
throughput_csma_non_persistent = np.zeros(numG)
throughput_1persistent = np.zeros(numG)
throughput_pt1persistent = np.zeros(numG)
throughput_pt01persistent = np.zeros(numG)
throughput_pt5persistent = np.zeros(numG)
throughput_csma_cd = np.zeros(numG)
G = np.logspace(-1.5, .9, numG)

for i, lam in enumerate(G):
    inter_arrival_time = rng.exponential(1 / lam, N)
    arrival_times = np.array(inter_arrival_time)
    for j in range(1, len(arrival_times)):
        arrival_times[j] += arrival_times[j - 1]
    throughput_pure[i] = pure_aloha(
        inter_arrival_time) / math.ceil(arrival_times[-1])
    throughput_slotted[i] = slotted_aloha(
        arrival_times) / math.ceil(arrival_times[-1])
    throughput_csma_non_persistent[i] = csma_generalized(
        arrival_times, 0) / math.ceil(arrival_times[-1])
    throughput_1persistent[i] = csma_generalized(
        arrival_times, 1) / math.ceil(arrival_times[-1])
    throughput_pt1persistent[i] = csma_generalized(
        arrival_times, .1) / math.ceil(arrival_times[-1])
    throughput_pt01persistent[i] = csma_generalized(
        arrival_times, .01) / math.ceil(arrival_times[-1])
    throughput_pt5persistent[i] = csma_generalized(
        arrival_times, .5) / math.ceil(arrival_times[-1])
    throughput_csma_cd[i] = csma_generalized(
        arrival_times, 1, True) / math.ceil(arrival_times[-1])

#### 5.2.6 Plot access-protocol throughput


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
theLineWidth = 2

ax.cla()
ax.axis([None, None, None, None])

ax.plot(G, 100*throughput_pure,
        linewidth=theLineWidth+2, label='Pure Aloha')
ax.plot(G, 100*throughput_slotted,
        linewidth=theLineWidth+2, label='Slotted Aloha')
ax.plot(G, 100*throughput_csma_non_persistent,
        linewidth=theLineWidth, label="CSMA non-persistent")
ax.plot(G, 100*throughput_1persistent,
        linewidth=theLineWidth, label="CSMA 1-persistent")
ax.plot(G, 100*throughput_pt1persistent,
        linewidth=theLineWidth, label="CSMA .1-persistent")
ax.plot(G, 100*throughput_pt5persistent,
        linewidth=theLineWidth, label="CSMA .5-persistent")
ax.plot(G, 100*throughput_pt01persistent,
        linewidth=theLineWidth, label="CSMA .01-persistent")
ax.plot(G, 100*throughput_csma_cd,
        linewidth=theLineWidth, label="CSMA Collision Detection")

plt.grid(True)

ax.set_facecolor((.2, 0.2, 0.3))
plt.legend(fancybox=True, framealpha=1,
           loc="upper right", facecolor=(.6, 0.61, 0.8))

plt.ylim(0, 100)

ax.set_xlabel('G (attempt rate)')
ax.set_ylabel('Throughput [%]')

plt.tight_layout()

The horizontal axis shows $G$, the mean number of generated transmission attempts per packet duration. For example, $G=0.5$ means one attempt every two packet durations on average. In an analytical ALOHA model, attempts include new transmissions and retries; this code generates the combined attempt stream directly.

For slotted ALOHA,

$$
S=G e^{-G}.
$$

A slot succeeds when exactly one attempt occurs. With Poisson arrivals, that probability is $G e^{-G}$. Equivalently, an individual attempt succeeds when no competing attempt occurs in its slot, with probability $e^{-G}$.

The vertical axis displays $100S$: utilization relative to one successful packet per packet duration. It is not normalized to each protocol's own maximum. Slotted ALOHA reaches its analytical maximum at $G=1$, where $S=1/e\approx0.368$. Random sampling and the simplified simulation rules can cause differences from the analytical curves.


----
### 5.3 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**Why can waiting for each ACK limit Stop-and-Wait throughput? What makes the protocol useful despite this limitation?**

Answer:


**What are the maximum throughputs of pure ALOHA and slotted ALOHA in the idealized models? How does slot alignment explain the difference?**

Answer:


**How do collision detection and collision avoidance differ? Why did shared, half-duplex Ethernet use CSMA/CD, while Wi-Fi uses CSMA/CA?**

Answer:


**A sender has transmitted packets 1–10. Packet 4 is lost, and packets 5–10 arrive out of order. Assuming packets 1–3 are acknowledged, what must be retransmitted under basic Go-Back-N and under Selective Repeat? What extra receiver capability does Selective Repeat require?**

Answer:


**Devices A and C can both reach receiver B, but cannot hear each other. If C starts transmitting while A is already sending to B, what can happen at B? Why does carrier sensing fail to prevent this?**

Answer:


**OPTIONAL: Are all explanations and exercises in this notebook clear? Which parts, if any, need clarification or improvement?**

Answer:
